### RAG end to end pipeline ###

In [ ]:
import tqdm
import os
from pathlib import Path
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Function to process PDF documents and extract text
def process_documents(doc_dir):
    doc_dir = "../data"
    all_documents = []
    pdf_directory = Path(doc_dir)
    pdf_files = list(pdf_directory.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf in pdf_files:
        print(f"Processing {pdf}...")

        try:
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()

            ##adding the metadata to the documents

            for doc in documents:
                doc.metadata["source"] = pdf.name

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf}: {e}")   

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents  
path="../data"
all_documents = process_documents(path)


In [ ]:
### text splitting the documents into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    all_chunks = []
    for doc in documents:
        chunks = text_splitter.split_documents([doc])
        all_chunks.extend(chunks)

    print(f"Total chunks created: {len(all_chunks)}")
    return all_chunks

split_documents(all_documents)

### Embedding and VectorStore DB ###

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
